<a href="https://colab.research.google.com/github/khoirulanamid/Upscale_Real_ESRGAN/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table>
  <tr>
    <td><img src="https://github.com/xinntao/Real-ESRGAN/raw/master/assets/realesrgan_logo.png" width="120"></td>
    <td><h1>4K Video Upscaler Colab (Real-ESRGAN)</h1></td>
  </tr>
</table>

[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/khoirulanamid/Upscale_Real_ESRGAN)
[![Real-ESRGAN](https://img.shields.io/badge/AI-Real--ESRGAN-orange)](https://github.com/xinntao/Real-ESRGAN)

---

### 🚀 **Selamat datang di Alat Upscale Media AI**
Notebook ini dirancang untuk mempermudah Anda meningkatkan resolusi **Video & Foto** secara otomatis hingga kualitas **4K** menggunakan algoritma **Real-ESRGAN** yang canggih.

**Author:** [khoirulanamid](https://github.com/khoirulanamid)  
**Credits:** Adapted from the original [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN) project.

---

# 1. Setup (~1 minute)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

!pip install -q torch==2.0.1 torchvision==0.15.2 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

!pip install "numpy<2"
mount_drive = False

### 🛠️ Perbaiki Masalah Kompatibilitas
Jalankan sel ini untuk menambal library `basicsr` dan memperbaiki error impor `torchvision`.  

**Penjelasan:** Bagian ini bertujuan untuk **memperbaiki masalah kompatibilitas** yang mungkin muncul pada beberapa library Python yang digunakan. Secara khusus, `basicsr` (salah satu library yang dipakai oleh Real-ESRGAN) kadang memiliki cara mengimpor `torchvision` yang sudah usang dan bisa menyebabkan error. Kode di sel ini akan **memodifikasi file `basicsr` secara otomatis** agar menggunakan perintah import yang benar dan terbaru, sehingga:
- **Mencegah error** yang tidak perlu.
- **Memastikan semua fungsi di notebook berjalan lancar** dan kompatibel dengan versi library yang terinstal.

In [ ]:
import os

# Path to the file causing the error
file_path = '/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    # Replace the outdated import with the compatible one
    new_content = content.replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale'
    )

    with open(file_path, 'w') as f:
        f.write(new_content)
    print("✅ basicsr/data/degradations.py patched successfully!")
else:
    print("❌ Could not find the file to patch. Please ensure the setup cell was run.")

# 2. Hubungkan ke Google Drive (Opsional)
Hubungkan Colab ke Drive agar Anda bisa mengambil file video dari Drive atau menyimpan hasil upscale langsung ke sana secara permanen.

*Jika tidak diaktifkan, hasil akan tersimpan di folder sementara dan akan terhapus jika sesi Colab berakhir.*

In [ ]:
# @title Klik untuk Menghubungkan Drive
from google.colab import drive
mount_drive = True #@param{type:"boolean"}

if mount_drive:
    print("⌛ Menghubungkan ke Google Drive...")
    drive.mount('/content/gdrive/')
    print("✅ Terhubung! Anda sekarang bisa menggunakan path '/content/gdrive/MyDrive/'")
else:
    print("⚠️ Drive tidak dihubungkan. Output akan disimpan di penyimpanan sementara.")

# 3. Jalankan Upscale (Video atau Foto)
Pilih mode yang ingin Anda gunakan, masukkan path file, lalu jalankan sel ini.

In [ ]:
import os, cv2
from PIL import Image

# @title 3. Panel Kendali Upscale (Batch Mode)
mode = "Video" #@param ["Video", "Foto"]

# @markdown ---
# @markdown ### 📁 LOKASI INPUT & OUTPUT
# @markdown *Masukkan path ke file tunggal ATAU folder untuk batch processing*
input_path = "/content/gdrive/MyDrive/Upscale_Input" #@param{type:"string"}
output_dir = "/content/gdrive/MyDrive/Upscale_Output" #@param{type:"string"}

# @markdown ---
# @markdown ### 📹 KONFIGURASI VIDEO
resolution = "4k (3840 x 2160)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"]
model_video = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]
output_fps = 60 #@param {type:"number"}
# @markdown *Masukkan 0 untuk mempertahankan FPS asli. Jika tidak, masukkan FPS yang diinginkan (misalnya, 30, 60).*
delete_input_video = True #@param {type:"boolean"}
# @markdown *Pilih apakah akan menghapus file video input setelah berhasil di-upscale.*

# @markdown ---
# @markdown ### 🖼️ KONFIGURASI FOTO
model_foto = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B"]
output_dpi = 300 #@param {type:"number"}
# @markdown *Masukkan 0 untuk mempertahankan DPI asli gambar (jika ada, default 96 DPI jika tidak ada). Masukkan nilai lain (misalnya, 300) untuk kualitas cetak standar.*
output_format = "PNG" #@param ["PNG", "JPG"]
apply_srgb = True #@param {type:"boolean"}
# @markdown *Output sRGB akan memastikan profil warna standar jika gambar Anda akan digunakan untuk tampilan web atau digital.*
delete_input_foto = True #@param {type:"boolean"}
# @markdown *Pilih apakah akan menghapus file foto input setelah berhasil di-upscale.*

def process_video(v_path):
    if not os.path.exists(v_path): return
    video_capture = cv2.VideoCapture(v_path)
    video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    original_fps = video_capture.get(cv2.CAP_PROP_FPS) # Get original FPS
    video_capture.release()

    if video_width == 0: return

    final_width, final_height = 1920, 1080
    if "2k" in resolution: final_width, final_height = 2560, 1440
    elif "4k" in resolution: final_width, final_height = 3840, 2160
    elif "2 x" in resolution: final_width, final_height = 2*video_width, 2*video_height
    elif "3 x" in resolution: final_width, final_height = 3*video_width, 3*video_height
    elif "4 x" in resolution: final_width, final_height = 4*video_width, 4*video_height

    scale_factor = max(final_width/video_width, final_height/video_height)
    print(f"🎬 Memproses Video: {os.path.basename(v_path)} ({video_width}x{video_height}) @ {original_fps:.2f} FPS")

    # Use absolute path to the script
    script_path = "/content/Real-ESRGAN/inference_realesrgan_video.py"
    !python {script_path} -n {model_video} -i '{v_path}' -o '{output_dir}' --outscale {scale_factor}

    video_name = os.path.splitext(os.path.basename(v_path))[0]
    potential_out = os.path.join(output_dir, f"{video_name}_out.mp4") # This is the output from Real-ESRGAN

    if os.path.exists(potential_out):
        final_v_upscaled_path = os.path.join(output_dir, f"{video_name}_upscaled.mp4")

        # Check if output_fps is specified and different from original_fps
        if output_fps > 0 and output_fps != original_fps:
            temp_output_path = os.path.join(output_dir, f"{video_name}_temp_fps.mp4")
            print(f"⚙️ Mengatur ulang FPS ke {output_fps} untuk {os.path.basename(v_path)}")
            # Use ffmpeg to change FPS
            !ffmpeg -i '{potential_out}' -filter:v fps=fps={output_fps} -c:a copy '{temp_output_path}' -y

            if os.path.exists(temp_output_path):
                if os.path.exists(final_v_upscaled_path): os.remove(final_v_upscaled_path)
                os.rename(temp_output_path, final_v_upscaled_path)
                os.remove(potential_out) # Remove the intermediate file from Real-ESRGAN
                print(f"✅ FPS berhasil diatur ulang untuk {os.path.basename(v_path)}")
            else:
                print(f"❌ Gagal mengatur ulang FPS untuk {os.path.basename(v_path)}, menggunakan FPS asli.")
                if os.path.exists(final_v_upscaled_path): os.remove(final_v_upscaled_path)
                os.rename(potential_out, final_v_upscaled_path) # Use original FPS output if re-encoding failed
        else:
            if os.path.exists(final_v_upscaled_path): os.remove(final_v_upscaled_path)
            os.rename(potential_out, final_v_upscaled_path) # Directly rename if no FPS change needed

        if delete_input_video:
            print(f"🗑️ Menghapus file input: {v_path}")
            os.remove(v_path)

def process_image(img_path, model_foto, output_dir, output_dpi_param, output_format, apply_srgb, delete_input_foto):
    if not os.path.exists(img_path): return
    print(f"🖼️ Memproses Foto: {os.path.basename(img_path)}")
    script_path = "/content/Real-ESRGAN/inference_realesrgan.py"

    # Real-ESRGAN will output a PNG by default
    !python {script_path} -n {model_foto} -i '{img_path}' -o '{output_dir}' --outscale 4

    # Determine the output path from Real-ESRGAN (it usually adds '_out' and keeps original extension if not specified)
    # Real-ESRGAN usually outputs to a folder named after the model, and then the image with _out
    # Let's assume it directly outputs to output_dir with _out.png based on previous code
    img_name_base = os.path.splitext(os.path.basename(img_path))[0]
    # Real-ESRGAN saves the output image as `filename_out.png` within the output folder
    realesrgan_output_path = os.path.join(output_dir, f"{img_name_base}_out.png")

    if os.path.exists(realesrgan_output_path):
        try:
            img = Image.open(realesrgan_output_path)

            final_dpi = output_dpi_param
            if output_dpi_param == 0: # User wants original DPI
                # Try to get original DPI from the image
                if hasattr(img, 'info') and 'dpi' in img.info:
                    final_dpi = img.info['dpi'][0] # PIL returns dpi as a tuple (x_dpi, y_dpi)
                    print(f"⚙️ Menggunakan DPI asli gambar: {final_dpi} untuk {os.path.basename(img_path)}")
                else:
                    final_dpi = 96 # Default DPI if not found
                    print(f"⚠️ DPI asli tidak ditemukan, menggunakan default 96 DPI untuk {os.path.basename(img_path)}")

            if apply_srgb and img.mode != 'RGB':
                img = img.convert('RGB')
                print(f"⚙️ Menerapkan konversi sRGB untuk {os.path.basename(img_path)}")

            # Construct final output filename
            final_img_name = f"{img_name_base}_upscaled.{output_format.lower()}"
            final_img_path = os.path.join(output_dir, final_img_name)

            if output_format == "JPG":
                # Save as JPG with specified DPI
                img.save(final_img_path, dpi=(final_dpi, final_dpi), quality=95) # Quality for JPG
                print(f"✅ Foto {os.path.basename(img_path)} berhasil di upscale dan disimpan sebagai JPG ({final_dpi} DPI).")
            else: # PNG
                # Save as PNG with specified DPI (less common but PIL supports it)
                img.save(final_img_path, dpi=(final_dpi, final_dpi)) # DPI for PNG is metadata
                print(f"✅ Foto {os.path.basename(img_path)} berhasil di upscale dan disimpan sebagai PNG ({final_dpi} DPI).")

            # Remove the intermediate file from Real-ESRGAN
            os.remove(realesrgan_output_path)

            if delete_input_foto:
                print(f"🗑️ Menghapus file input: {img_path}")
                os.remove(img_path)

        except Exception as e:
            print(f"❌ Gagal memproses output gambar dengan PIL: {e}")
            print(f"⚠️ Menyimpan output asli dari Real-ESRGAN: {realesrgan_output_path}")
    else:
        print(f"❌ Output dari Real-ESRGAN tidak ditemukan untuk {os.path.basename(img_path)}")

os.makedirs(output_dir, exist_ok=True)
files_to_process = []

if os.path.isdir(input_path):
    extensions = (".mp4", ".mkv", ".mov", ".avi") if mode == "Video" else (".jpg", ".jpeg", ".png", ".webp")
    files_to_process = [os.path.join(input_path, f) for f in os.listdir(input_path) if f.lower().endswith(extensions)]
else:
    files_to_process = [input_path]

print(f"📦 Batch Processing: Ditemukan {len(files_to_process)} file.")

for file in files_to_process:
    if mode == "Video":
        process_video(file)
    else:
        process_image(file, model_foto, output_dir, output_dpi, output_format, apply_srgb, delete_input_foto)

print("✅ Semua proses batch selesai!")

# 4. Putuskan Koneksi Otomatis (Opsional)
Aktifkan ini jika Anda ingin sesi Colab otomatis berhenti setelah proses selesai. Ini sangat berguna untuk:
*   **Menghemat Kuota GPU**: Tidak membuang-buang jatah pemakaian saat Anda sedang tidak di depan komputer.
*   **Otomatisasi**: Anda bisa menjalankan proses lalu ditinggal tidur; sistem akan menutup sendiri.

*⚠️ Pastikan Anda sudah menyimpan hasil ke Google Drive (Mount Drive aktif) sebelum menggunakan fitur ini agar file tidak hilang.*

In [ ]:
from google.colab import runtime

disconnect_when_finish = True  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()